# Módulo 04 · Clasificación y pruebas de hipótesis — Soluciones

*Material para instructores.* Este notebook es autocontenido: reproduce la preparación mínima del notebook del módulo
(datos, partición train/test, modelos y experimento A/B simulado con la misma semilla) y resuelve los cuatro ejercicios finales.

| Ejercicio | Tema | Secciones del notebook original |
|---|---|---|
| 1 | Umbral óptimo con costes FN/FP de 5 y 50 | §3 |
| 2 | Random forest con las 5 variables más importantes | §2 y §4 |
| 3 | Tamaño muestral para 10 % → 10,5 % y duración | §5 |
| 4 | IC 95 % de la diferencia y mejora relativa | §6 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower

sns.set_theme(style="whitegrid")

## Preparación (igual que en el notebook)
Maligno = 1, partición 70/30 estratificada con `random_state=0`.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = 1 - data.target  # 1 = maligno
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
rf = RandomForestClassifier(300, random_state=0).fit(X_train, y_train)

cv = StratifiedKFold(5, shuffle=True, random_state=0)
prob_cv = cross_val_predict(logit, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prob_test = logit.predict_proba(X_test)[:, 1]
print(f"Train: {len(y_train)} casos | Test: {len(y_test)} casos ({y_test.sum()} malignos)")

## Ejercicio 1 · ¿Cómo cambia el umbral óptimo con el coste FN/FP?

Reutilizamos las probabilidades de validación cruzada sobre el *train* (para no elegir el umbral mirando el test)
y buscamos el umbral que minimiza el coste total para tres ratios: 5, 20 (el del notebook) y 50.

In [ ]:
umbrales = np.linspace(0.01, 0.99, 99)

def coste_total(prob, y_real, u, coste_fn, coste_fp=1):
    fn = ((prob < u) & (y_real == 1)).sum()
    fp = ((prob >= u) & (y_real == 0)).sum()
    return coste_fn * fn + coste_fp * fp

resultados = []
fig, ax = plt.subplots(figsize=(7, 4))
for ratio in [5, 20, 50]:
    costes = np.array([coste_total(prob_cv, y_train.values, u, ratio) for u in umbrales])
    u_opt = umbrales[np.argmin(costes)]
    ax.plot(umbrales, costes / costes.max(), label=f"FN/FP = {ratio} (óptimo {u_opt:.2f})")
    ax.axvline(u_opt, ls="--", lw=0.8, c=ax.lines[-1].get_color())
    tn, fp, fn, tp = confusion_matrix(y_test, (prob_test >= u_opt).astype(int)).ravel()
    resultados.append({"FN/FP": ratio, "umbral óptimo (CV)": round(u_opt, 2), "FN test": fn, "FP test": fp,
                       "recall test": round(tp / (tp + fn), 3), "precisión test": round(tp / (tp + fp), 3),
                       "coste test": ratio * fn + fp})
ax.set_xlabel("umbral"); ax.set_ylabel("coste relativo (CV)"); ax.legend(); ax.set_title("Coste según el umbral")
plt.show()

# Referencia: umbral 0,5 por defecto
tn, fp, fn, tp = confusion_matrix(y_test, (prob_test >= 0.5).astype(int)).ravel()
print(f"Umbral 0,5 en test: FN = {fn}, FP = {fp}")
tabla_umbral = pd.DataFrame(resultados).set_index("FN/FP")
tabla_umbral

**Interpretación.** Cuanto más caro es un falso negativo respecto a un falso positivo, más bajo es el umbral óptimo:
el modelo pasa a marcar como «sospechoso» cualquier caso con una probabilidad pequeña de malignidad.
- Con **FN/FP = 5** el umbral óptimo en CV es **0,45**, casi el 0,5 por defecto: en test pasamos de 4 FN y 4 FP a 3 FN y 5 FP.
- Con **FN/FP = 20** cae a **0,06**: en test queda 1 solo falso negativo, a cambio de 18 falsos positivos (precisión 0,78).
- Con **FN/FP = 50** el umbral **no cambia** (0,06): con este conjunto de datos el modelo separa tan bien las clases que,
  por debajo de ese punto, ya no quedan malignos «escondidos» que rescatar en el train; bajar más el listón solo añadiría
  falsos positivos.

La regla teórica, si las probabilidades estuvieran perfectamente calibradas, sería $u^* = \frac{C_{FP}}{C_{FP}+C_{FN}}$:
1/6 ≈ 0,17 para un ratio de 5, 1/21 ≈ 0,05 para 20 y 1/51 ≈ 0,02 para 50. Los valores empíricos no coinciden exactamente
(con 398 casos la curva de coste es escalonada y casi plana en amplias zonas, y la regresión logística da probabilidades
muy extremas en este problema), pero la dirección es la misma. Moraleja: el umbral se elige con datos y costes, no con una fórmula ciega.

**Explicación para un gerente de hospital (3 frases):**
> «El modelo da a cada paciente una probabilidad de tumor maligno, y nosotros decidimos a partir de qué probabilidad
> pedimos una biopsia. Cuanto más grave consideramos no detectar un cáncer frente a hacer una biopsia innecesaria, más bajo
> ponemos ese listón: así casi no se nos escapa ningún caso, a cambio de más pruebas a pacientes sanas. Con un coste 20 veces
> mayor para el cáncer no detectado, el listón baja del 50 % al 6 % y solo se nos escapa 1 de 64 tumores en la prueba,
> a cambio de 18 biopsias innecesarias de 107 casos benignos; el equipo clínico debe validar ese ratio de costes, porque es una
> decisión médica y económica, no técnica.»

## Ejercicio 2 · Random forest con solo las 5 variables más importantes

Seleccionamos las 5 variables con mayor importancia **calculada con el train** (así no hay fuga de información del test)
y comparamos el AUC en test y en validación cruzada.

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top5 = importancias.index[:5].tolist()
print("Top 5 variables:", top5)

rf5 = RandomForestClassifier(300, random_state=0).fit(X_train[top5], y_train)
auc_30 = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
auc_5 = roc_auc_score(y_test, rf5.predict_proba(X_test[top5])[:, 1])

auc_cv_30 = cross_val_score(RandomForestClassifier(300, random_state=0), X_train, y_train, cv=cv, scoring="roc_auc")
auc_cv_5 = cross_val_score(RandomForestClassifier(300, random_state=0), X_train[top5], y_train, cv=cv, scoring="roc_auc")

comparacion = pd.DataFrame({
    "nº variables": [30, 5],
    "AUC test": [auc_30, auc_5],
    "AUC CV (media)": [auc_cv_30.mean(), auc_cv_5.mean()],
    "AUC CV (desv.)": [auc_cv_30.std(), auc_cv_5.std()],
}, index=["RF completo", "RF top 5"]).round(4)
print(f"Diferencia de AUC (completo − top 5): test {auc_30 - auc_5:+.4f} | CV {auc_cv_30.mean() - auc_cv_5.mean():+.4f}")
comparacion

In [ ]:
print("Correlaciones entre las 5 variables elegidas:")
X_train[top5].corr().round(2)

**Interpretación.** En el conjunto de test el AUC es prácticamente idéntico con 5 variables que con 30 (0,980 en ambos
casos; la diferencia, 0,0004, es incluso a favor del modelo reducido). La validación cruzada sobre el train, que es una
estimación más estable, muestra una pérdida pequeña: de 0,994 a 0,980 (≈1,4 centésimas), con algo más de variabilidad
entre pliegues. Conclusión: se pierde muy poca capacidad de ordenar casos.

¿Merece la pena? En este contexto, **probablemente sí**: cinco medidas son más fáciles de obtener, de auditar y de explicar
a un comité clínico, y el modelo es menos frágil ante cambios en el protocolo de medición. Dos matices:
- Las 5 variables están muy correlacionadas entre sí (todas miden tamaño o concavidad del núcleo). Esa redundancia explica
  por qué se pierde tan poco, y también por qué la importancia de cada una por separado no debe leerse como causal.
- La decisión final no debe basarse solo en el AUC: conviene comparar también el recall al umbral de negocio elegido en el
  ejercicio 1, que es lo que de verdad importa en un cribado.

## Ejercicio 3 · Tamaño muestral para detectar 10 % → 10,5 %

In [ ]:
analisis = NormalIndPower()
for p1, p2 in [(0.10, 0.11), (0.10, 0.105)]:
    h = proportion_effectsize(p2, p1)
    n = np.ceil(analisis.solve_power(effect_size=h, alpha=0.05, power=0.8, alternative="two-sided"))
    total = 2 * n
    dias = np.ceil(total / 3000)
    print(f"{p1:.1%} → {p2:.1%}: h = {h:.4f} | n por grupo = {n:,.0f} | total = {total:,.0f} | días con 3.000 visitas/día = {dias:.0f}")

**Interpretación.** Reducir a la mitad el efecto que queremos detectar (de +1 a +0,5 puntos) multiplica el tamaño
muestral aproximadamente por **cuatro**, porque $n \propto 1/\delta^2$. Con 3.000 visitas diarias repartidas entre A y B
(1.500 por grupo), el experimento necesitaría **39 días** (57.756 usuarios por grupo, 115.512 en total), frente a
10 días para detectar +1 punto (14.745 por grupo).

En la práctica, además, conviene **redondear a semanas completas** (para cubrir el ciclo semanal de comportamiento):
6 semanas en lugar de 39 días. Y preguntarse si un efecto de +0,5 puntos merece el coste de oportunidad de tener el
experimento en marcha mes y medio.
Alternativas: aceptar un efecto mínimo detectable mayor, usar técnicas de reducción de varianza (p. ej. CUPED) o
aumentar el tráfico dedicado.

## Ejercicio 4 · IC 95 % de la diferencia y mejora relativa

Regeneramos el experimento con la misma semilla y el mismo orden de simulación que en el notebook (§6), así que los
datos son idénticos.

In [ ]:
rng = np.random.default_rng(3)
n = 15_000
conv_A = rng.binomial(1, 0.100, n)
conv_B = rng.binomial(1, 0.112, n)
xA, xB = conv_A.sum(), conv_B.sum()
pA, pB = xA / n, xB / n
diff = pB - pA

z, p = proportions_ztest([xB, xA], [n, n])

# IC de Wald (no agrupado) "a mano"
ee = np.sqrt(pA * (1 - pA) / n + pB * (1 - pB) / n)
ic_wald = (diff - 1.96 * ee, diff + 1.96 * ee)
# IC con statsmodels (método de Newcombe, más robusto)
ic_newc = confint_proportions_2indep(xB, n, xA, n, method="newcomb", compare="diff")

# Mejora relativa (lift) y su IC: método delta sobre log(pB/pA) y bootstrap como comprobación
lift = pB / pA - 1
ee_log = np.sqrt((1 - pA) / (n * pA) + (1 - pB) / (n * pB))
ic_lift_delta = (np.exp(np.log(pB / pA) - 1.96 * ee_log) - 1, np.exp(np.log(pB / pA) + 1.96 * ee_log) - 1)

rng_boot = np.random.default_rng(0)
bA = rng_boot.binomial(n, pA, 10_000) / n
bB = rng_boot.binomial(n, pB, 10_000) / n
ic_lift_boot = np.percentile(bB / bA - 1, [2.5, 97.5])

print(f"Conversión A = {pA:.2%} | B = {pB:.2%}")
print(f"Diferencia B − A = {diff:+.2%} (puntos porcentuales)")
print(f"  IC 95 % Wald:     [{ic_wald[0]:+.2%}, {ic_wald[1]:+.2%}]")
print(f"  IC 95 % Newcombe: [{ic_newc[0]:+.2%}, {ic_newc[1]:+.2%}]")
print(f"Mejora relativa = {lift:+.1%}")
print(f"  IC 95 % (delta):     [{ic_lift_delta[0]:+.1%}, {ic_lift_delta[1]:+.1%}]")
print(f"  IC 95 % (bootstrap): [{ic_lift_boot[0]:+.1%}, {ic_lift_boot[1]:+.1%}]")
print(f"z = {z:.2f}, p-valor = {p:.2g}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.2))
ax.errorbar(diff * 100, 0, xerr=[[(diff - ic_newc[0]) * 100], [(ic_newc[1] - diff) * 100]], fmt="o", capsize=6)
ax.axvline(0, c="grey", ls="--", label="sin efecto")
ax.axvline(1, c="tab:red", ls=":", label="efecto mínimo de interés (+1 p.p.)")
ax.set_yticks([]); ax.set_xlabel("diferencia de conversión B − A (puntos porcentuales)")
ax.legend(loc="upper left", fontsize=8); ax.set_title("IC 95 % de la diferencia")
plt.tight_layout(); plt.show()

**Interpretación.**
- La conversión de B supera a la de A en torno a **+1,6 puntos** (de ≈9,9 % a ≈11,5 %), una **mejora relativa de ≈+16 %**.
- El IC 95 % de la diferencia va de **+0,87 a +2,27 puntos**: no incluye el 0, coherente con un p-valor
  muy pequeño (≈1,1·10⁻⁵). El IC de la mejora relativa va de ≈+8,5 % a ≈+23,7 % (delta y bootstrap coinciden).
- Lectura correcta del p-valor: *si* A y B convirtieran igual, sería muy improbable observar una diferencia tan grande
  como esta por puro azar. **No** significa «99,99 % de probabilidad de que B sea mejor».
- El extremo inferior del IC (≈+0,9 p.p.) queda cerca del efecto mínimo de interés que usamos para planificar (+1 p.p.):
  el efecto es real con mucha confianza, pero su tamaño exacto es incierto. El escenario pesimista sigue siendo positivo.

**Recomendación:** lanzar B, porque incluso en el extremo bajo del intervalo la mejora es positiva y el coste de cambiar
un botón es pequeño.

**Qué más querría saber antes de decidir:**
1. Si el tamaño muestral (15.000 por grupo) se fijó **antes** de empezar y no se paró al ver el resultado (*peeking*).
2. Si la conversión era la **métrica principal** declarada de antemano, y qué pasa con las métricas de salvaguarda
   (ticket medio, devoluciones, tiempo de carga). Un botón más llamativo podría aumentar compras pequeñas o devoluciones.
3. Si el reparto A/B fue correcto (comprobación de *sample ratio mismatch*) y si el efecto se mantiene a lo largo de las
   semanas (efecto novedad) y en los segmentos principales (móvil/escritorio, clientes nuevos/recurrentes), sin convertir
   esto en una pesca de segmentos significativos.
4. El impacto económico: +1,6 p.p. de conversión multiplicado por el tráfico y el margen medio, frente al coste de implantación.